# Learner Activity: Retail Performance Regression

Use the retail performance dataset to repeat the correlation and simple linear regression workflow from the lecture notebook.


In [103]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

DATA_FILE = "retail-performance-activity-data.csv"


## 1. Load And Inspect The Data

Load the CSV, preview the first few rows, and identify which columns are numeric.


In [104]:
df = pd.read_csv("retail-performance-activity-data.csv")
df.head()


,month,campaign_name,ad_spend,store_visits,transactions,avg_order_value,discount_rate,sales_revenue
0,2025-01,New Year Sale,18000,42000,3150,42.5,0.12,134800
1,2025-02,Loyalty Push,19500,43800,3295,43.1,0.10,141900
2,2025-03,Spring Refresh,21000,46100,3460,44.0,0.11,152600
3,2025-04,Weekend Deals,22500,47200,3515,44.7,0.13,156100
4,2025-05,Member Month,24100,49800,3720,45.2,0.09,168400


In [105]:
# Identify the numeric columns you want to compare.
# Use the answer key column order: ad_spend, store_visits, transactions,
# avg_order_value, discount_rate, and sales_revenue.
numeric_columns = [
    "ad_spend",
    "store_visits",
    "transactions",
    "avg_order_value",
    "discount_rate",
    "sales_revenue"
]

# Use the numeric_columns list to select only those columns from df.
# Store the result in a DataFrame named numeric_df.
numeric_df = df[numeric_columns]
numeric_df.head()


,ad_spend,store_visits,transactions,avg_order_value,discount_rate,sales_revenue
0,18000,42000,3150,42.5,0.12,134800
1,19500,43800,3295,43.1,0.10,141900
2,21000,46100,3460,44.0,0.11,152600
3,22500,47200,3515,44.7,0.13,156100
4,24100,49800,3720,45.2,0.09,168400


## 2. Manual Pearson Correlation

Write a function that manually calculates Pearson correlation between two numeric arrays. Then use it to compare each numeric column against `sales_revenue`.


In [106]:
def pearson_correlation(x: np.ndarray, y: np.ndarray) -> float:
    # Convert x and y to numeric NumPy arrays before doing math.
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # Calculate the mean of x and the mean of y.
    x_mean = np.mean(x)
    y_mean = np.mean(y)

    # Use the centered values to calculate the Pearson numerator.
    numerator = np.sum((x - x_mean) * (y - y_mean))

    # Use the centered values to calculate the Pearson denominator.
    denominator = np.sqrt(
        np.sum((x - x_mean) ** 2) *
        np.sum((y - y_mean) ** 2)
    )

    # Return the Pearson correlation coefficient.
    return numerator / denominator

In [107]:
# Identify the target column you want to compare every numeric column against.
target = "sales_revenue"
target_values = numeric_df[target].to_numpy(dtype=float)

# Use an empty dictionary to collect each column's correlation with sales_revenue.
target_correlations = {}

# Loop through each numeric column.
for column in numeric_df.columns:
    # Identify the x values from the current column.
    column_values = numeric_df[column].to_numpy(dtype=float)

    # Use the pearson_correlation function.
    # Pass the current column values as x and sales_revenue values as y.
    correlation = pearson_correlation(column_values, target_values)

    # Store the result in the dictionary using the column name as the key.
    target_correlations[column] = correlation

# Convert the dictionary to a Series and sort from strongest to weakest.
target_correlation_series = pd.Series(target_correlations).sort_values(ascending=False)
target_correlation_series


sales_revenue      1.000000
transactions       0.999819
store_visits       0.999672
ad_spend           0.994390
avg_order_value    0.986387
discount_rate      0.594815
dtype: float64

## 3. Manual Correlation Matrix

Build a correlation matrix manually by looping through every pair of numeric columns.


In [108]:
def build_correlation_matrix(numeric_df: pd.DataFrame) -> pd.DataFrame:
    # Create an empty square DataFrame.
    # Use the numeric column names as both the row index and column labels.
    correlation_matrix = pd.DataFrame(index=numeric_df.columns, columns=numeric_df.columns, dtype=float)

    # Loop through each row variable.
    for row in numeric_df.columns:
        # Loop through each column variable.
        for column in numeric_df.columns:
            # Identify the two arrays to compare.
            row_values = numeric_df[row].to_numpy(dtype=float)
            column_values = numeric_df[column].to_numpy(dtype=float)

            # Use pearson_correlation to calculate the relationship.
            correlation = pearson_correlation(row_values, column_values)

            # Store the result where the row variable and column variable meet.
            correlation_matrix.loc[row, column] = correlation

    return correlation_matrix

# Call the function and display the first few rows.
correlation_matrix = build_correlation_matrix(numeric_df)
correlation_matrix.head()


,ad_spend,store_visits,transactions,avg_order_value,discount_rate,sales_revenue
ad_spend,1.000000,0.995688,0.994455,0.997803,0.556802,0.994390
store_visits,0.995688,1.000000,0.999882,0.988408,0.596984,0.999672
transactions,0.994455,0.999882,1.000000,0.986441,0.597476,0.999819
avg_order_value,0.997803,0.988408,0.986441,1.000000,0.540518,0.986387
discount_rate,0.556802,0.596984,0.597476,0.540518,1.000000,0.594815


In [109]:
# Build a Plotly heatmap of your manual correlation matrix.
# Identify the z values, x labels, and y labels from correlation_matrix.
fig = go.Figure(
    data=go.Heatmap(
        z=correlation_matrix,
        x=correlation_matrix.columns,
        y=correlation_matrix.index,
        zmin=-1,
        zmax=1,
        colorscale="RdYlGn",
        text=np.round(correlation_matrix.to_numpy(dtype=float), 3),
        texttemplate="%{text}",
    )
)
fig.update_layout(title= "Correlation Matrix", template="plotly_white")
fig.show()


## 4. Simple Linear Regression

Choose one useful predictor for `sales_revenue`, then manually calculate a simple linear regression model.


In [110]:
# Identify the predictor column.
# Use ad_spend first so your output can be compared with the answer key.
predictor = "ad_spend"

# Identify the predictor values as x and sales revenue as y.
x = df[predictor].to_numpy(dtype=float)
y = df[target].to_numpy(dtype=float)

# Use pearson_correlation to check the relationship between x and y.
correlation = pearson_correlation(x, y)
correlation


np.float64(0.9943898580946783)

In [111]:
def simple_linear_regression(x: np.ndarray, y: np.ndarray) -> dict:
    # Convert x and y to numeric NumPy arrays.
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # Calculate the mean of x and the mean of y.
    x_mean = x.mean()
    y_mean = y.mean()

    # Calculate the slope.
    slope = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)

    # Calculate the intercept.
    intercept = y_mean - (slope * x_mean)

    # Generate predictions using the regression equation.
    predictions = intercept + slope * x

    # Calculate residuals, SSE, SST, R2, and RMSE.
    # Residuals
    residuals = y - predictions

    #sse
    sse = np.sum(residuals ** 2)

    #sst
    sst = np.sum((y - y_mean) ** 2)

    #r_squared
    
    r_squared = 1 - (sse / sst)
    
    #rmse
    rmse = np.sqrt(np.mean(residuals ** 2))

    # Return all results in a dictionary so later cells can use them by name.
    return {
        "slope": slope,
        "intercept": intercept,
        "predictions": predictions,
        "sse": sse,
        "sst": sst,
        "r_squared": r_squared,
        "rmse": rmse,
    }


In [112]:
# Run the simple_linear_regression function.
# Pass the predictor array as x and sales revenue as y.
result = simple_linear_regression(x, y)

print("Simple Linear Regression Results")
print("--------------------------------")
print("Model Coefficients")
print("------------------")
print(f"Slope: {result["slope"]}")
print(f"Intercept: {result["intercept"]}")
print(f"Equation: sales_revenue = {result["intercept"]} + ({result["slope"]} * ad_spend)")
print()
print("Model Evaluation Metrics")
print("------------------------")
print(f"SSE: {result["sse"]}")
print(f"SST: {result["sst"]}")
print(f"R2 (fit metric): {result["r_squared"]}")
print(f"RMSE (error metric): {result["rmse"]}")
print()
print("Relationship Metric")
print("-------------------")
print(f"Pearson r (ad_spend vs sales_revenue): {correlation}")


Simple Linear Regression Results
--------------------------------
Model Coefficients
------------------
Slope: 7.089176310415248
Intercept: 681.2003630587715
Equation: sales_revenue = 681.2003630587715 + (7.089176310415248 * ad_spend)

Model Evaluation Metrics
------------------------
SSE: 225552500.56727916
SST: 20158756666.666668
R2 (fit metric): 0.9888111898815546
RMSE (error metric): 4335.440198404302

Relationship Metric
-------------------
Pearson r (ad_spend vs sales_revenue): 0.9943898580946783


In [113]:
# Sort the data by the predictor so the regression line is drawn from left to right.
plot_df = df.sort_values(predictor).copy()

# Match the prediction order to the sorted predictor values.
sorted_indices = np.argsort(x)

plot_df["predicted_sales_revenue"] = result["predictions"][sorted_indices]

# Create a scatter plot of the actual data.
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=plot_df[predictor],
        y=plot_df["sales_revenue"],
        mode="markers",
        name="Actual data",
        text=plot_df["campaign_name"],
        hovertemplate="Campaign: %{text}<br>ad_spend: %{x}<br>sales_revenue: %{y}<extra></extra>",
    )
)

# Add a regression line using the predicted sales revenue values.
fig.add_trace(
    go.Scatter(
        x=plot_df[predictor],
        y=plot_df["predicted_sales_revenue"],
        mode="lines",
        name="Regression line",
        hovertemplate="ad_spend: %{x}<br>predicted sales_revenue: %{y}<extra></extra>",
    )
)

# Label the chart and axes clearly.
fig.update_layout(
    title="Ad Spend vs Sales Revenue",
    xaxis_title="Ad Spend",
    yaxis_title="Sales Revenue",
    template="plotly_white",
)

fig.show()

In [114]:
# Record the predictions in the original DataFrame.
df["predicted_sales_revenue"] = result["predictions"]

# Save the DataFrame with predictions to results.csv.
df.to_csv()


',month,campaign_name,ad_spend,store_visits,transactions,avg_order_value,discount_rate,sales_revenue,predicted_sales_revenue\r\n0,2025-01,New Year Sale,18000,42000,3150,42.5,0.12,134800,128286.37395053323\r\n1,2025-02,Loyalty Push,19500,43800,3295,43.1,0.1,141900,138920.1384161561\r\n2,2025-03,Spring Refresh,21000,46100,3460,44.0,0.11,152600,149553.90288177898\r\n3,2025-04,Weekend Deals,22500,47200,3515,44.7,0.13,156100,160187.66734740185\r\n4,2025-05,Member Month,24100,49800,3720,45.2,0.09,168400,171530.34944406626\r\n5,2025-06,Midyear Clearance,25800,52100,3890,46.1,0.15,179300,183581.94917177217\r\n6,2025-07,Back To School Preview,27200,54500,4075,46.8,0.12,190100,193506.7960063535\r\n7,2025-08,Back To School Peak,28900,57800,4320,47.2,0.14,203900,205558.39573405945\r\n8,2025-09,Payday Promo,30500,59200,4430,48.0,0.1,212500,216901.07783072384\r\n9,2025-10,Holiday Warmup,32200,61800,4625,48.7,0.11,225300,228952.67755842977\r\n10,2025-11,Black Friday,34800,68100,5110,49.5,0.18,253200,

## 5. Interpretation

Write 2 to 4 sentences answering:

- Which variable did you use to predict sales revenue?
- ANSWER: The variable I used to predict sales revenue was ad_spend. It was selected as the independent variable in the simpler linear regression model.
- How strong is the relationship?
- ANSWER: The strength of the relationship is determined through the Pearson correlation coefficient and R2 value. A value cloer to 1 indicates a strong positive linear relationship, while a value closer to 0 indicates a weak realtionship.
- What does the slope mean in business terms?
- ANSWER: The slope represents the exepcted change in sales revenue for every one unit increase in ad spend. In business terms, it quantifies how much additional revenue is generated per unit of advertising investment, assuming other factors remain constant.
- Is this model useful for decision-making? Why or why not?
- ANSWER: The model is useful for decisio-making if the R2 calue is high, because it shows that ad spend explains variation in sales revenue. However, if the value is low, it is not useful because it only explains a small portion of the variation in sales revenue.

## Bonus Challenge: Lagged Ad Spend

Calculate the correlation between `ad_spend` and future `sales_revenue` using 0-month, 1-month, 2-month, and 3-month lags. Which lag has the strongest relationship?


In [115]:
# Display the current-month ad_spend values before creating a lag.
non_lagged = df["ad_spend"]
non_lagged


0     18000
1     19500
2     21000
3     22500
4     24100
5     25800
6     27200
7     28900
8     30500
9     32200
10    34800
11    36500
Name: ad_spend, dtype: int64

In [116]:
# Use the shift function to move ad_spend down by 1 row.
# This lets you compare earlier ad_spend with later sales_revenue.
lagged = df["ad_spend"].shift()
lagged


0         NaN
1     18000.0
2     19500.0
3     21000.0
4     22500.0
5     24100.0
6     25800.0
7     27200.0
8     28900.0
9     30500.0
10    32200.0
11    34800.0
Name: ad_spend, dtype: float64

In [117]:
# Use an empty dictionary to store each lag's correlation result.
lag_results = {}

# Loop through 0-month, 1-month, 2-month, and 3-month lags.
for lag in range(4):
    # Create lagged ad_spend values and drop NaN rows caused by shifting
    lagged_ad_spend = df["ad_spend"].shift(lag).iloc[lag:].to_numpy(dtype=float)

    # Align sales_revenue to match the shifted data
    aligned_sales_revenue = df["sales_revenue"].iloc[lag:].to_numpy(dtype=float)

    # Compute Pearson correlation
    lag_r = pearson_correlation(lagged_ad_spend, aligned_sales_revenue)

    # Store result with clear label
    lag_results[f"lag_{lag}_months"] = lag_r

# Convert dictionary to Series and sort
lag_series = pd.Series(lag_results).sort_values(ascending=False)
lag_series

lag_0_months    0.994390
lag_1_months    0.991886
lag_3_months    0.991215
lag_2_months    0.989354
dtype: float64

Write 1 to 2 sentences answering: Which lag has the strongest relationship with `sales_revenue`? What does that suggest about current-month ad spend versus earlier ad spend?


In [118]:
# Plot the lagged correlations as a bar chart.
fig = go.Figure(
    data=go.Bar(
        x=lag_series.index,
        y=lag_series.values,
        text=np.round(lag_series.values, 3),
        textposition="auto",
    )
)
fig.update_layout(title= "Lagged Ad Spend vs Sales Revenue Correlation",  
                xaxis_title="Lag (Months)",
                yaxis_title="Pearson Correlation",
                template="plotly_white")
fig.show()


# Bonus Info on Lagged Relationships

Lagged relationships can be useful for forecasting, but they are not necessarily causal. Think about whether both ad spend and sales revenue might be moving together because of seasonality, growth, or promotions.
